# Papagun — подготовка и проверка в Jupyter
Весь рабочий код этапа подготовки находится в этом notebook: загрузка Excel, DataFrame, расчёт модели пациентов, отрисовка, экспорт PPTX, регрессионные проверки и тест 1/3/6/11/18 филиалов. Проектные `.py` и `.ps1` не нужны. GUI и выделение модулей выполняются позже.

Запускайте ячейки сверху вниз из корня проекта. Нужны pandas, openpyxl и python-pptx 1.0.2; для самого интерфейса Jupyter — notebook/ipykernel в выбранном окружении. Например: `python -m pip install pandas openpyxl python-pptx==1.0.2 notebook`.
Исходники не изменяются, существующие результаты не перезаписываются. Перед сохранением notebook очищайте outputs: финансовые данные и локальные пути не должны сохраняться в его результатах.


## Зависимости выбранного ядра
Пакет называется `python-pptx`, а импортируется как `pptx`. Пакеты должны быть установлены в то Python-окружение, которое выбрано ядром notebook.

Если отсутствует `pptx`, выполните отдельную ячейку `%pip install python-pptx==1.0.2`, затем перезапустите ядро и выполните notebook сверху вниз. `%pip` устанавливает пакет в окружение текущего ядра.

Ниже проверяются три рабочие библиотеки. По умолчанию сеть не используется. Для установки отсутствующих пакетов этой же ячейкой установите `INSTALL_MISSING_DEPENDENCIES = True`, выполните её и при необходимости перезапустите ядро. Уже установленные пакеты не обновляются.

In [ ]:
import sys
import subprocess
import importlib.util
import importlib

INSTALL_MISSING_DEPENDENCIES = False
required_packages = {
    'pandas': 'pandas',
    'openpyxl': 'openpyxl',
    'pptx': 'python-pptx==1.0.2',
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    install_command = '%pip install ' + ' '.join(missing_packages)
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError(
            'В выбранном ядре отсутствуют зависимости. Выполните отдельную ячейку: '
            + install_command
            + '. Затем перезапустите ядро и выполните notebook сверху вниз. '
            + 'Или включите INSTALL_MISSING_DEPENDENCIES в этой ячейке.'
        )
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
    importlib.invalidate_caches()
    if any(importlib.util.find_spec(module) is None for module in required_packages):
        raise RuntimeError('Установка завершена. Перезапустите ядро и выполните notebook сверху вниз.')
print('Зависимости выбранного ядра доступны.')


In [ ]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
from io import BytesIO
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import hashlib
import json
import os
import re
import subprocess

import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from pptx import Presentation
from pptx.chart.data import CategoryChartData
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE
from pptx.opc.constants import RELATIONSHIP_TYPE as RT
from pptx.opc.package import _Relationship
from pptx.opc.packuri import PackURI
from pptx.util import Inches, Pt


## Настройки запуска
По умолчанию создаются один рабочий слайд и тестовая презентация на 5 слайдов; проверки включены. `OUTPUT_DIR` задаёт папку результатов. Повторный запуск выбирает свободное имя. `RENDER_PREVIEWS=False` оставляет в output только PPTX; необязательные превью показываются из временного каталога и удаляются после просмотра.


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'excel_loading.ipynb').is_file():
    raise FileNotFoundError('Запустите ядро notebook из корня проекта.')
OUTPUT_DIR = ROOT / 'output'
TEMPLATE = ROOT / 'Исходники/Презентация_NL_9 мес 2026.pptx'
CENTERS = ROOT / 'Исходники/Соответствие названий центров в презентации и эксель Nephrolife.xlsx'
REPORT_MONTH = 'сентябрь'
BRANCH_COUNTS = (1, 3, 6, 11, 18)
GENERATE_PRESENTATIONS = True
RUN_CHECKS = True
RENDER_PREVIEWS = False


## Исходные книги

Читаются `NL_Кол. Бюджет.xlsx` и `NL_Фин. Бюджет.xlsx` из папки `Исходники`.
Запускайте ядро из корня проекта; при другом расположении измените `DATA_DIR` на папку с этими книгами.


In [ ]:
DATA_DIR = Path.cwd() / 'Исходники'
excel_files = {
    'financial': DATA_DIR / 'NL_Фин. Бюджет.xlsx',
    'quantity': DATA_DIR / 'NL_Кол. Бюджет.xlsx',
}

missing_files = [key for key, path in excel_files.items() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Не найдены книги: ' + ', '.join(missing_files)
        + '. Проверьте DATA_DIR и имена файлов.'
    )


## Загрузка всех листов

`sheet_name=None` возвращает словарь листов. `header=None` сохраняет все строки, поскольку строка заголовков ещё не согласована. `dtype=object` отключает подбор типа для всей колонки; `keep_default_na=False` сохраняет текстовые маркеры вроде `NA`, а пустые ячейки обычно представлены пустой строкой. Пропуски не заменяются нулями.

Это первичное чтение значений, а не полная модель Excel: оформление и объединения не сохраняются в DataFrame, ошибки ячеек могут стать пропусками. Формулы не пересчитываются — читаются сохранённые Excel результаты, которые могут отсутствовать или быть устаревшими. Эти случаи требуют отдельной проверки перед расчётами.

In [ ]:
workbooks = {
    key: pd.read_excel(
        path,
        sheet_name=None,
        header=None,
        dtype=object,
        keep_default_na=False,
        engine='openpyxl',
    )
    for key, path in excel_files.items()
}

financial_sheets = workbooks['financial']
quantity_sheets = workbooks['quantity']

In [ ]:
sheet_summary = pd.DataFrame([
    {'book': key, 'sheet': sheet, 'rows': frame.shape[0], 'columns': frame.shape[1]}
    for key, sheets in workbooks.items()
    for sheet, frame in sheets.items()
])
sheet_summary

## Отдельные DataFrame филиалов двух книг NL

`quantity_df` — количественный бюджет, `financial_df` — финансовый бюджет.
Для каждой книги применяется прежняя методика: одна строка исходной статьи × филиал × месяц,
шесть колонок: филиал, статья, месяц, утверждено, факт по норме, факт по 1С.
Книги не объединяются, показатели не пересчитываются и не округляются.

Используются те же 20 листов филиалов, строки 9–141 количественной книги и 9–193 финансовой.
Заголовки статей (C6), месяцы (строка 6) и три показателя (строка 7) проверяются.
Сводные и вспомогательные листы, АУП, кварталы и годовые колонки исключены.
Итоги и повторяющиеся статьи сохраняются; пустые названия пропускаются.
Пустые показатели представлены `pd.NA`, нули сохранены, ошибки Excel восстановлены как текстовые коды.
Формулы не пересчитываются: используются сохранённые результаты Excel.


In [ ]:
BRANCH_SHEETS = [
    'Актобе', 'Актобе АМЦ', 'Астана Биржан Сал', 'Астана Сарыарка',
    'Атырау', 'Атырау Нурсая', 'Атырау Нурсая 2', 'Караганда ЦАД',
    'Караганда Строителей', 'Кентау', 'Костанай', 'Лисаковск', 'Рудный',
    'Павлодар Дюсенова', 'Павлодар Павлова', 'Экибастуз', 'Петропавловск',
    'Усть-Каменогорск (NL UK)', 'Шымкент', 'Шымкент Абая',
]
ARTICLE_ROWS = {'financial': (9, 193), 'quantity': (9, 141)}  # номера строк Excel, включительно
MONTHS = [
    'январь', 'февраль', 'март', 'апрель', 'май', 'июнь',
    'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь',
]
RESULT_COLUMNS = [
    'Наименование филиала', 'Наименование статей', 'Месяц',
    'Утверждено', 'Факт по норме', 'Факт по 1С',
]


In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


def header_text(value):
    # Нормализация применяется только к заголовкам для их проверки.
    return ' '.join(str(value).split()).casefold()


def build_branches_dataframe(source_book):
    if source_book not in ARTICLE_ROWS:
        raise ValueError("Источник должен быть 'financial' или 'quantity'.")
    sheets = workbooks[source_book]
    missing = [sheet for sheet in BRANCH_SHEETS if sheet not in sheets]
    if missing:
        raise ValueError('Не найдены листы филиалов: ' + ', '.join(missing))

    first_row, last_row = ARTICLE_ROWS[source_book]
    records, origins, issues = [], [], []
    # pandas превращает ошибки ячеек Excel в NaN. Второе чтение позволяет
    # восстановить коды ошибок, не смешивая их с пустыми значениями.
    cached_book = load_workbook(excel_files[source_book], read_only=True, data_only=True)
    try:
        for branch in BRANCH_SHEETS:
            frame = sheets[branch]
            if frame.shape[0] < last_row or frame.shape[1] < 56:
                raise ValueError(f'Недостаточный размер основной таблицы: {branch}')
            if header_text(frame.iat[5, 2]) != 'наименование статей':
                raise ValueError(f'Не найден заголовок статей в C6: {branch}')

            month_columns = {}
            for column, value in enumerate(frame.iloc[5]):
                month = header_text(value)
                if month not in MONTHS:
                    continue
                if month in month_columns:
                    raise ValueError(f'Повторяющийся заголовок месяца: {branch}, {month}')
                expected = ['утвержд.', 'факт по норме', 'факт по 1с']
                actual = [header_text(frame.iat[6, column + offset]) for offset in range(3)]
                if actual != expected:
                    raise ValueError(f'Неожиданные показатели месяца: {branch}, {month}')
                month_columns[month] = column
            if set(month_columns) != set(MONTHS):
                raise ValueError(f'Ожидаются все 12 месяцев: {branch}')

            cached_rows = cached_book[branch].iter_rows(
                min_row=first_row, max_row=last_row, max_col=frame.shape[1],
            )
            for excel_row, cells in enumerate(cached_rows, start=first_row):
                article = frame.iat[excel_row - 1, 2]
                if cells[2].data_type == 'e':
                    raise ValueError(f'Ошибка Excel вместо названия статьи: {branch}, C{excel_row}')
                if pd.isna(article) or article == '' or (isinstance(article, str) and not article.strip()):
                    continue
                if not isinstance(article, str):
                    raise ValueError(f'Название статьи не является текстом: {branch}, C{excel_row}')
                for month in MONTHS:
                    column = month_columns[month]
                    values = []
                    coordinates = []
                    for offset, metric in enumerate(RESULT_COLUMNS[3:]):
                        coordinate = f'{get_column_letter(column + offset + 1)}{excel_row}'
                        coordinates.append(coordinate)
                        cell = cells[column + offset]
                        value = frame.iat[excel_row - 1, column + offset]
                        if cell.data_type == 'e':
                            value = cell.value
                            issues.append({
                                'Строка DataFrame': len(records), 'Филиал': branch,
                                'Месяц': month, 'Показатель': metric,
                                'Ячейка': coordinate, 'Ошибка Excel': value,
                            })
                        elif pd.isna(value) or value == '':
                            value = pd.NA
                        values.append(value)
                    records.append([branch, article, month, *values])
                    origins.append([source_book, branch, excel_row, *coordinates])
    finally:
        cached_book.close()

    result = pd.DataFrame(records, columns=RESULT_COLUMNS, dtype=object)
    provenance = pd.DataFrame(origins, columns=[
        'Книга', 'Лист', 'Строка Excel',
        'Ячейка Утверждено', 'Ячейка Факт по норме', 'Ячейка Факт по 1С',
    ])
    diagnostics = pd.DataFrame(issues, columns=[
        'Строка DataFrame', 'Филиал', 'Месяц', 'Показатель', 'Ячейка', 'Ошибка Excel',
    ])
    return result, provenance, diagnostics


quantity_df, quantity_provenance, quantity_diagnostics = build_branches_dataframe('quantity')
financial_df, financial_provenance, financial_diagnostics = build_branches_dataframe('financial')

# Совместимость с прежними ячейками исследования количественного бюджета.
branches_df = quantity_df
branches_provenance = quantity_provenance
branches_diagnostics = quantity_diagnostics


In [ ]:
# Только размеры и диагностика, без значений показателей.
dataframe_summary = pd.DataFrame([
    {
        'Книга': key,
        'Листов': len(workbooks[key]),
        'Строк': len(frame),
        'Филиалов': frame['Наименование филиала'].nunique(),
        'Месяцев': frame['Месяц'].nunique(),
        'Ошибок Excel': len(diagnostics),
    }
    for key, frame, diagnostics in [
        ('quantity', quantity_df, quantity_diagnostics),
        ('financial', financial_df, financial_diagnostics),
    ]
])
dataframe_summary


## Доступ к результатам

Основные таблицы: `quantity_df` и `financial_df`.
Координаты источников с тем же индексом: `quantity_provenance` и `financial_provenance`.
Ошибки Excel: `quantity_diagnostics` и `financial_diagnostics`.
Все исходные листы доступны отдельно в `quantity_sheets` и `financial_sheets`.

`branches_df`, `branches_provenance`, `branches_diagnostics` оставлены как ссылки на количественный бюджет.
Для локального просмотра можно выполнить `quantity_df.head()` или `financial_df.head()` в отдельной ячейке.
Перед сохранением удаляйте результаты с содержимым таблиц.
Не суммируйте итоговые строки вместе с детализацией без согласованного правила.
При смене версии книг перепроверьте список филиалов и границы `ARTICLE_ROWS`.


## Модель пациентов и форматирование
Сентябрь — численность на конец 9 месяцев, без суммирования месяцев. Строка Excel 14, план «Утверждено», факт «Факт по 1С»; координата отличает одноимённые статьи. Сопоставление центров берётся из справочника. Формулы, проверки и форматирование перенесены из прототипа без изменения методики.


In [ ]:
def _center_key(text):
    key = re.sub(r'[^а-яёa-z0-9]', '', str(text).casefold().replace('\\n', ''))
    return key.replace('12мкрн', '12мкр')


def _number(value):
    if pd.isna(value) or isinstance(value, (str, bool)):
        raise ValueError('Показатель пациентов отсутствует или не является числом.')
    result = Decimal(str(value))
    if not result.is_finite() or result < 0 or result != result.to_integral_value():
        raise ValueError('Ожидается целое неотрицательное число пациентов.')
    return result


def _percent(value):
    return str((value * 100).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP)).replace('.', ',') + '%'


def _integer(value):
    return f'{int(value):,}'.replace(',', ' ')


def _counted(value, forms):
    number = abs(int(value))
    if 11 <= number % 100 <= 14:
        word = forms[2]
    elif number % 10 == 1:
        word = forms[0]
    elif 2 <= number % 10 <= 4:
        word = forms[1]
    else:
        word = forms[2]
    return f'{_integer(value)} {word}'


def patient_model(quantity_df, quantity_provenance, labels, *, month):
    """Experimental snapshot: source row 14, approved vs actual 1C; no month sums."""
    if month not in MONTHS:
        raise ValueError('Неизвестный месяц.')
    if not quantity_df.index.is_unique or not quantity_df.index.equals(quantity_provenance.index):
        raise ValueError('Индексы DataFrame и происхождения не совпадают или не уникальны.')
    if not quantity_provenance['Книга'].eq('quantity').all():
        raise ValueError('Ожидается происхождение количественного бюджета.')
    wb = load_workbook(CENTERS, read_only=True, data_only=True)
    try:
        mapping = {}
        for title, branch in list(wb.active.values)[1:]:
            if not title or not branch:
                continue
            # Explicit suffix difference between supplied directory and workbook.
            if branch == 'Усть-Каменогорск':
                branch = 'Усть-Каменогорск (NL UK)'
            key = _center_key(title)
            if key in mapping:
                raise ValueError('Повторяющееся название в справочнике центров.')
            mapping[key] = branch
    finally:
        wb.close()
    rows = []
    for label in labels:
        branch = mapping.get(_center_key(label))
        if branch is None:
            raise ValueError('Центр образца не найден в справочнике.')
        selected = quantity_df.loc[
            quantity_df['Наименование филиала'].eq(branch)
            & quantity_df['Месяц'].eq(month)
            & quantity_provenance['Строка Excel'].eq(14)
        ]
        if len(selected) != 1:
            raise ValueError('Для центра требуется ровно одна строка пациентов.')
        row = selected.iloc[0]
        origin = quantity_provenance.loc[selected.index[0]]
        if origin['Лист'] != branch or row['Наименование статей'].strip() != 'Количество пациентов':
            raise ValueError('Изменилась структура источника: строка 14 должна содержать пациентов.')
        plan, actual = _number(row['Утверждено']), _number(row['Факт по 1С'])
        if plan == 0:
            raise ValueError('Нулевой план: правило расчёта исполнения требует согласования.')
        rows.append(dict(label=label.strip(), branch=branch, plan=plan, actual=actual,
                         ratio=actual / plan, source_index=selected.index[0]))
    if len({r['branch'] for r in rows}) != len(rows):
        raise ValueError('Один филиал сопоставлен нескольким центрам.')
    # Original is sorted by execution; stable ordering resolves equal ratios.
    return sorted(rows, key=lambda r: r['ratio'], reverse=True)


## Отрисовка и экспорт одного слайда
Диаграмма редактируемая; ширина внутренней области пропорциональна количеству центров. Полная рамка сохраняется для корректного отображения чисел. Необязательный `template` позволяет тестировать копию образца с выбранным числом центров без изменения глобальных настроек.


In [ ]:
def build_slide2(quantity_df, quantity_provenance, *, month='сентябрь',
                 output=ROOT / 'output/NL_пациенты_2026_слайд2.pptx', template=None):
    """Use source styling; replace chart workbook, labels, title and all totals."""
    output = Path(output)
    if output.suffix.lower() != '.pptx':
        raise ValueError('Выходной файл должен иметь расширение .pptx.')
    if output.resolve().is_relative_to((ROOT / 'Исходники').resolve()):
        raise ValueError('Сохранение в каталог исходников запрещено.')
    if output.exists():
        raise FileExistsError('Результат уже существует; укажите новое имя файла.')
    deck = Presentation(TEMPLATE if template is None else template)
    slide = deck.slides[1]
    chart = next(shape.chart for shape in slide.shapes if shape.has_chart)
    labels = [cat.label for cat in chart.plots[0].categories]
    model = patient_model(quantity_df, quantity_provenance, labels, month=month)
    # Drop unused slide relationships so unrelated slide data are not exported.
    for sld_id in list(deck.slides._sldIdLst):
        if deck.part.related_slide(sld_id.rId) is not slide:
            deck.part.drop_rel(sld_id.rId)
            deck.slides._sldIdLst.remove(sld_id)
    for rel in list(slide.part.rels.values()):
        if rel.reltype.endswith('/notesSlide'):
            slide.part.drop_rel(rel.rId)
    data = CategoryChartData(number_format='0')
    data.categories = [r['label'] for r in model]
    data.add_series('План', [int(r['plan']) for r in model])
    data.add_series('Факт', [int(r['actual']) for r in model])
    chart.replace_data(data)
    # Keep the physical category pitch of the 18-center sample. Narrow only
    # the inner plot: a tiny chart frame makes PowerPoint wrap numeric labels.
    chart_shape = next(shape for shape in slide.shapes if shape.has_chart)
    layout = chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
    plot_fraction = .9666091006930311 * min(1, len(model) / 18)
    layout.xpath('./c:w')[0].set('val', str(plot_fraction))
    layout.xpath('./c:x')[0].set('val', str((1 - plot_fraction) / 2))
    for alignment in chart._chartSpace.xpath('.//c:catAx/c:lblAlgn'):
        alignment.set('val', 'ctr')
    chart.plots[0].gap_width = 70
    chart.plots[0].overlap = -10
    # The source uses a third bar series for manually moved percentages. Replace
    # it with editable badges; remove per-point overrides tied to old values.
    for series in chart.series:
        for node in series._element.xpath('./c:dLbls/c:dLbl | ./c:dPt | ./c:extLst | ./c:spPr/a:effectLst'):
            node.getparent().remove(node)
        for body in series._element.xpath('./c:dLbls/c:txPr/a:bodyPr'):
            body.set('wrap', 'none')
            body.set('lIns', '0')
            body.set('rIns', '0')
    plan = sum((r['plan'] for r in model), Decimal(0))
    actual = sum((r['actual'] for r in model), Decimal(0))
    title, summary = [s for s in slide.shapes if s.has_text_frame]
    title.text_frame.paragraphs[0].runs[2].text = f'ПОКАЗАТЕЛИ: {month} 2026 г.'
    title.text_frame.paragraphs[0].runs[4].text = ' '
    # The source wraps the title in leading/trailing soft line breaks, making
    # its text taller than the colored title box.
    for br in title.text_frame.paragraphs[0]._p.xpath('./a:br'):
        br.getparent().remove(br)
    title.text_frame.margin_top = title.text_frame.margin_bottom = 0
    title.height = Pt(26)
    pars = summary.text_frame.paragraphs
    pars[0].runs[0].text = f'{month.capitalize()} 2026: {_counted(len(model), ("центр", "центра", "центров"))} образца'
    pars[1].runs[1].text = _counted(plan, ('пациент', 'пациента', 'пациентов'))
    pars[2].runs[1].text = _counted(actual, ('пациент', 'пациента', 'пациентов'))
    difference = _counted(actual - plan, ('пациент', 'пациента', 'пациентов'))
    number, word = difference.rsplit(' ', 1)
    pars[3].runs[1].text = number + ' '
    pars[3].runs[2].text = word
    pars[4].runs[1].text = _percent(actual / plan)
    color = RGBColor.from_string('FF0000' if actual < plan else '4472C4')
    pars[3].runs[1].font.color.rgb = color
    pars[4].runs[1].font.color.rgb = color
    # Plot boundaries follow the inner manual layout of the sample.
    layout = chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
    plot_x = chart_shape.left + chart_shape.width * float(layout.xpath('./c:x')[0].get('val'))
    plot_w = chart_shape.width * float(layout.xpath('./c:w')[0].get('val'))
    for i, row in enumerate(model):
        center = plot_x + plot_w * (i + 0.5) / len(model)
        box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE,
                                    round(center - Inches(.28)), Inches(1.47),
                                    Inches(.56), Inches(.24))
        box.name = f'Исполнение {i+1}'
        box.fill.solid()
        box.fill.fore_color.rgb = RGBColor(255, 255, 255)
        box.line.color.rgb = RGBColor(0, 0, 0)
        box.line.width = Pt(.8)
        frame = box.text_frame
        frame.margin_left = frame.margin_right = 0
        frame.margin_top = frame.margin_bottom = 0
        frame.vertical_anchor = MSO_ANCHOR.MIDDLE
        p = frame.paragraphs[0]
        p.alignment = PP_ALIGN.CENTER
        r = p.add_run()
        r.text = _percent(row['ratio'])
        r.font.name, r.font.size, r.font.bold = 'Arial Narrow', Pt(12), True
        r.font.color.rgb = RGBColor.from_string('FF0000' if row['ratio'] < Decimal('.95') else '4472C4')
    # Remove unused sections/custom shows that may still refer to dropped slides.
    for node in deck.part._element.xpath('./p:custShowLst | ./p:extLst'):
        node.getparent().remove(node)
    output.parent.mkdir(parents=True, exist_ok=True)
    buffer = BytesIO()
    deck.save(buffer)
    with output.open('xb') as target:
        target.write(buffer.getvalue())
    return output


In [ ]:
def available_output(filename):
    """Choose a new result name without overwriting previous results."""
    path = Path(filename)
    suffix = 2
    candidate = path
    while candidate.exists():
        candidate = path.with_name(f'{path.stem}_{suffix}{path.suffix}')
        suffix += 1
    return candidate

slide2_path = None
if GENERATE_PRESENTATIONS:
    slide2_path = build_slide2(
        quantity_df, quantity_provenance,
        month=REPORT_MONTH,
        output=available_output(OUTPUT_DIR / 'NL_пациенты_2026_слайд2.pptx'),
    )


## Регрессионная проверка значений и ошибочных входов
Проверяются все 18 центров по независимому чтению AO14/AQ14, сортировка, встроенная XLSX, сводка и блокировки ошибок: пропуск, ошибка Excel, отрицательная/дробная численность, нулевой план, неверный месяц/индекс, перезапись результата и запись в исходник. Итоги теста не содержат значений показателей.


In [ ]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).digest()


def must_fail(exc, action):
    try:
        action()
    except exc:
        return
    raise AssertionError(f'Expected {exc.__name__}')


def verify_slide2(df, prov):
    sources = [TEMPLATE, CENTERS, *sorted((ROOT / 'Исходники').glob('NL_*.xlsx'))]
    before = {p: digest(p) for p in sources}
    nb = json.loads((ROOT / 'excel_loading.ipynb').read_text(encoding='utf-8'))
    original = Presentation(TEMPLATE)
    original_chart = next(s.chart for s in original.slides[1].shapes if s.has_chart)
    labels = [c.label for c in original_chart.plots[0].categories]
    model = patient_model(df, prov, labels, month='сентябрь')
    assert len(model) == 18
    assert len({r['branch'] for r in model}) == 18
    assert all(a['ratio'] >= b['ratio'] for a, b in zip(model, model[1:]))
    # Independent source read: AO/AQ = approved/actual 1C in September.
    wb = load_workbook(ROOT / 'Исходники/NL_Кол. Бюджет.xlsx', read_only=True, data_only=True)
    try:
        for r in model:
            cells = list(wb[r['branch']].iter_rows(min_row=14, max_row=14, max_col=43, values_only=True))[0]
            assert r['plan'] == cells[40] and r['actual'] == cells[42]
    finally:
        wb.close()
    idx = model[0]['source_index']
    for value in (pd.NA, '#DIV/0!', -1, 1.5):
        broken = df.copy()
        broken.at[idx, 'Факт по 1С'] = value
        must_fail(ValueError, lambda: patient_model(broken, prov, labels, month='сентябрь'))
    broken = df.copy()
    broken.at[idx, 'Утверждено'] = 0
    must_fail(ValueError, lambda: patient_model(broken, prov, labels, month='сентябрь'))
    # A second article with the same name must not be aggregated accidentally.
    duplicate_name = df.copy()
    other = prov['Строка Excel'].eq(26) & duplicate_name['Месяц'].eq('сентябрь')
    duplicate_name.loc[other, 'Наименование статей'] = 'Количество пациентов'
    duplicate_name.loc[other, 'Факт по 1С'] = 999999
    assert patient_model(duplicate_name, prov, labels, month='сентябрь') == model
    must_fail(ValueError, lambda: patient_model(df, prov.iloc[::-1], labels, month='сентябрь'))
    must_fail(ValueError, lambda: patient_model(df, prov, labels, month='неизвестно'))
    temporary = TemporaryDirectory(prefix='papagun-slide2-')
    target = Path(temporary.name) / 'slide2.pptx'
    build_slide2(df, prov, output=target)
    must_fail(FileExistsError, lambda: build_slide2(df, prov, output=target))
    must_fail(ValueError, lambda: build_slide2(df, prov, output=TEMPLATE))
    deck = Presentation(target)
    assert len(deck.slides) == 1
    chart = next(s.chart for s in deck.slides[0].shapes if s.has_chart)
    assert len(chart.series) == 2
    assert [c.label for c in chart.plots[0].categories] == [r['label'] for r in model]
    for i, field in enumerate(('plan', 'actual')):
        assert list(chart.series[i].values) == [float(r[field]) for r in model]
    assert sum(s.name.startswith('Исполнение ') for s in deck.slides[0].shapes) == 18
    text = '\n'.join(s.text for s in deck.slides[0].shapes if s.has_text_frame)
    plan, actual = sum(r['plan'] for r in model), sum(r['actual'] for r in model)
    assert 'План: ' + _integer(plan) + ' пациент' in text
    assert 'Факт: ' + _integer(actual) + ' пациент' in text
    assert _percent(actual / plan) in text
    with ZipFile(target) as package:
        assert len([n for n in package.namelist() if n.startswith('ppt/slides/slide') and n.endswith('.xml')]) == 1
        assert len([n for n in package.namelist() if n.startswith('ppt/charts/chart') and n.endswith('.xml')]) == 1
        embeddings = [n for n in package.namelist() if n.endswith('.xlsx')]
        assert len(embeddings) == 1
        embedded = load_workbook(BytesIO(package.read(embeddings[0])), read_only=True, data_only=True)
        try:
            entries = list(embedded.active.values)
            assert len(entries) == 19
            for entry, row in zip(entries[1:], model):
                assert entry[:3] == (row['label'], int(row['plan']), int(row['actual']))
        finally:
            embedded.close()
    assert all(digest(p) == before[p] for p in sources)
    assert all(c.get('outputs', []) == [] and c.get('execution_count') is None
               for c in nb['cells'] if c['cell_type'] == 'code')
    temporary.cleanup()
    print('PASS: 18 centers; independent source, chart, embedded XLSX, totals, failures, notebook hygiene, SHA-256.')


In [ ]:
if RUN_CHECKS:
    verify_slide2(quantity_df, quantity_provenance)
    for number, expected in [(0, '0 центров'), (1, '1 центр'), (2, '2 центра'),
                             (3, '3 центра'), (5, '5 центров'), (11, '11 центров'),
                             (14, '14 центров'), (21, '21 центр'), (22, '22 центра')]:
        assert _counted(number, ('центр', 'центра', 'центров')) == expected


## Проверка компоновки: 1, 3, 6, 11 и 18 филиалов
Ниже находится весь алгоритм теста: временные шаблоны, вызов общего генератора, проверка значений и геометрии, объединение частей PPTX, проверка XML и независимых XLSX. Пять вложенных наборов выбираются воспроизводимо, начиная с самых длинных подписей. Это технический тест; бизнес-фильтры не вводятся.


In [ ]:
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def require(condition, message):
    if not condition:
        raise AssertionError(message)


def percentage(value):
    return str((value * 100).quantize(Decimal('.1'), rounding=ROUND_HALF_UP)).replace('.', ',') + '%'


def integer(value):
    return f'{int(value):,}'.replace(',', ' ')


def join_slides(paths):
    """Copy already generated OPC parts without changing slide XML or drawing."""
    result = Presentation(paths[0])
    package = result.part.package
    layout = result.slides[0].slide_layout.part
    reserved = {str(p.partname) for p in package.iter_parts()}
    for path in paths[1:]:
        source = Presentation(path)
        copied = {}

        def copy_part(part):
            if part in copied:
                return copied[part]
            pattern = re.sub(r'\d+(?=\.[^.]+$)', '%d', str(part.partname))
            if '%d' not in pattern:
                stem, suffix = str(part.partname).rsplit('.', 1)
                pattern = stem + '%d.' + suffix
            index = 1
            while pattern % index in reserved:
                index += 1
            name = PackURI(pattern % index)
            reserved.add(str(name))
            clone = type(part).load(name, part.content_type, package, part.blob)
            copied[part] = clone
            for rel in part.rels.values():
                if rel.is_external:
                    target, mode = rel.target_ref, 'External'
                else:
                    target = layout if rel.reltype == RT.SLIDE_LAYOUT else copy_part(rel.target_part)
                    mode = 'Internal'
                # Preserve rIds, so slide/chart XML remains byte-for-byte intact.
                clone.rels._rels[rel.rId] = _Relationship(name.baseURI, rel.rId, rel.reltype, mode, target)
            return clone

        slide_part = copy_part(source.slides[0].part)
        rid = result.part.relate_to(slide_part, RT.SLIDE)
        result.slides._sldIdLst.add_sldId(rid)
    return result


def check_slide(slide, expected, width, height):
    charts = [s.chart for s in slide.shapes if s.has_chart]
    require(len(charts) == 1, 'Expected one chart.')
    chart = charts[0]
    require(len(chart.series) == 2, 'Expected plan and actual series.')
    require([c.label for c in chart.plots[0].categories] == [r['label'] for r in expected], 'Wrong categories/order.')
    for series, field in zip(chart.series, ('plan', 'actual')):
        require(list(series.values) == [float(r[field]) for r in expected], 'Wrong chart values.')
    workbook = load_workbook(BytesIO(chart.part.chart_workbook.xlsx_part.blob), read_only=True, data_only=True)
    try:
        entries = list(workbook.active.values)
        require(len(entries) == len(expected) + 1, 'Stale rows in embedded workbook.')
        for entry, row in zip(entries[1:], expected):
            require(entry[:3] == (row['label'], int(row['plan']), int(row['actual'])), 'Wrong embedded values.')
    finally:
        workbook.close()
    plan = sum((r['plan'] for r in expected), Decimal(0))
    actual = sum((r['actual'] for r in expected), Decimal(0))
    summary = next(s for s in slide.shapes if s.has_text_frame and 'образца' in s.text)
    paragraphs = summary.text_frame.paragraphs
    count_match = re.search(r'(\d+)\s+центр(?:а|ов)?\s+образца', paragraphs[0].text)
    require(count_match is not None and int(count_match[1]) == len(expected), 'Wrong center count.')
    for paragraph, prefix, value in zip(paragraphs[1:4], ('План: ', 'Факт: ', 'Отклонение: '), (plan, actual, actual - plan)):
        require(re.fullmatch(re.escape(prefix + integer(value)) + r' пациент(?:а|ов)?', paragraph.text) is not None,
                'Wrong plan/actual/difference total.')
    require(paragraphs[4].text == 'Исполнение ' + percentage(actual / plan), 'Wrong total execution.')
    badges = sorted((s for s in slide.shapes if s.name.startswith('Исполнение ')), key=lambda s: s.left)
    require(len(badges) == len(expected), 'Wrong badge count.')
    for badge, row in zip(badges, expected):
        require(badge.text == percentage(row['ratio']), 'Wrong execution badge.')
    require(all(a.left + a.width <= b.left for a, b in zip(badges, badges[1:])), 'Execution badges overlap.')
    chart_shape = next(s for s in slide.shapes if s.has_chart)
    # Regression guard: physical chart width per center stays below one inch,
    # and the reduced chart is centered rather than stretched across the slide.
    layout = chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
    plot_width = chart_shape.width * float(layout.xpath('./c:w')[0].get('val'))
    plot_left = chart_shape.left + chart_shape.width * float(layout.xpath('./c:x')[0].get('val'))
    require(plot_width / len(expected) <= 914400, 'Small selection stretches columns.')
    require(abs(plot_left + plot_width / 2 - width / 2) <= 2, 'Compact plot is not centered.')
    require(chart._chartSpace.xpath('.//c:catAx/c:lblAlgn')[0].get('val') == 'ctr', 'Category labels are not centered.')
    for shape in slide.shapes:
        require(shape.left >= 0 and shape.top >= 0 and shape.left + shape.width <= width
                and shape.top + shape.height <= height, 'Shape extends beyond slide bounds.')


In [ ]:
def run_branch_count_test(counts, output, *, quantity_df, quantity_provenance):
    """Use DataFrames already loaded in this notebook; generate and verify five slides."""
    output = Path(output)
    require(len(counts) == 5 and len(set(counts)) == 5, 'Specify exactly five distinct branch counts.')
    require(any(n % 2 for n in counts) and any(n % 2 == 0 for n in counts), 'Include odd and even counts.')
    require(not output.exists(), 'Output already exists; select another --output.')
    require(output.suffix.lower() == '.pptx', 'Expected a PPTX output.')
    require(not output.resolve().is_relative_to((ROOT / 'Исходники').resolve()), 'Cannot write to sources.')
    sources = [TEMPLATE, CENTERS, ROOT / 'excel_loading.ipynb',
               *sorted((ROOT / 'Исходники').glob('NL_*.xlsx'))]
    before = {p: sha(p) for p in sources}
    require(quantity_df is not None and quantity_provenance is not None, 'Supply DataFrame and provenance.')
    df, provenance = quantity_df, quantity_provenance
    source = Presentation(TEMPLATE)
    chart = next(s.chart for s in source.slides[1].shapes if s.has_chart)
    labels = [c.label for c in chart.plots[0].categories]
    require(all(1 <= n <= len(labels) for n in counts), 'Counts must be in the range of source centers (1..18).')
    # Reproducible nesting, starting with long labels, not random patient data.
    order = sorted(range(len(labels)), key=lambda i: (-len(labels[i].strip()), i))
    all_model = patient_model(df, provenance, labels, month='сентябрь')
    by_label = {r['label']: r for r in all_model}
    # Expected numbers are read separately from Excel, rather than reusing the renderer's sums.
    book = load_workbook(ROOT / 'Исходники/NL_Кол. Бюджет.xlsx', read_only=True, data_only=True)
    try:
        for row in all_model:
            values = next(book[row['branch']].iter_rows(min_row=14, max_row=14, max_col=43, values_only=True))
            require(row['plan'] == values[40] and row['actual'] == values[42], 'DataFrame differs from source.')
    finally:
        book.close()
    with TemporaryDirectory(prefix='papagun-counts-') as temporary:
        folder = Path(temporary)
        generated, expectations = [], []
        for number, count in enumerate(counts, 1):
            indices = order[:count]
            chosen = [labels[i] for i in indices]
            expected = sorted((by_label[label.strip()] for label in chosen), key=lambda r: r['ratio'], reverse=True)
            fixture = Presentation(TEMPLATE)
            fixture_chart = next(s.chart for s in fixture.slides[1].shapes if s.has_chart)
            data = CategoryChartData()
            data.categories = chosen
            for series in chart.series:
                data.add_series(series.name, [series.values[i] for i in indices])
            fixture_chart.replace_data(data)
            template = folder / f'template_{count}.pptx'
            fixture.save(template)
            target = folder / f'slide_{count}.pptx'
            # Only the test input is swapped; actual calculation and rendering are invoked unchanged.
            build_slide2(df, provenance, output=target, template=template)
            result = Presentation(target)
            check_slide(result.slides[0], expected, result.slide_width, result.slide_height)
            generated.append(target)
            expectations.append(expected)
            print(f'PASS slide {number}: {count} centers, chart/workbook/totals/badges/bounds.')
        combined = join_slides(generated)
        buffer = BytesIO()
        combined.save(buffer)
        final = Presentation(BytesIO(buffer.getvalue()))
        require(len(final.slides) == 5, 'Expected five combined slides.')
        for slide, expected, single_path in zip(final.slides, expectations, generated):
            check_slide(slide, expected, final.slide_width, final.slide_height)
            original_slide = Presentation(single_path).slides[0]
            require(slide._element.xml == original_slide._element.xml, 'Assembly changed the drawing XML.')
            require(next(s.chart for s in slide.shapes if s.has_chart)._chartSpace.xml
                    == next(s.chart for s in original_slide.shapes if s.has_chart)._chartSpace.xml,
                    'Assembly changed chart XML.')
        with ZipFile(BytesIO(buffer.getvalue())) as archive:
            require(archive.testzip() is None, 'Invalid ZIP package.')
            require(len([n for n in archive.namelist() if n.startswith('ppt/charts/chart') and n.endswith('.xml')]) == 5,
                    'Charts must be independent for every slide.')
            require(len([n for n in archive.namelist() if n.endswith('.xlsx')]) == 5, 'Expected five independent workbooks.')
        require(all(sha(p) == before[p] for p in sources), 'Source or production algorithm was modified.')
        output.parent.mkdir(parents=True, exist_ok=True)
        with output.open('xb') as target:
            target.write(buffer.getvalue())
    print('PASS: five-slide deck; compact layout; assembly preserves XML; source and notebook SHA-256 unchanged during test.')
    return output


In [ ]:
test_pptx_path = None
test_summary = []
if GENERATE_PRESENTATIONS:
    test_pptx_path = available_output(OUTPUT_DIR / 'NL_test_branch_counts_fixed.pptx')
    run_branch_count_test(
        BRANCH_COUNTS, test_pptx_path,
        quantity_df=quantity_df, quantity_provenance=quantity_provenance,
    )
    test_deck = Presentation(test_pptx_path)
    assert len(test_deck.slides) == len(BRANCH_COUNTS)
    for number, (slide, expected_count) in enumerate(zip(test_deck.slides, BRANCH_COUNTS), 1):
        chart_shape = next(shape for shape in slide.shapes if shape.has_chart)
        actual_count = len(chart_shape.chart.plots[0].categories)
        assert actual_count == expected_count
        plot_fraction = float(chart_shape.chart._chartSpace.xpath(
            './/c:plotArea/c:layout/c:manualLayout/c:w')[0].get('val'))
        assert chart_shape.width * plot_fraction / actual_count <= 914400
        test_summary.append({'Слайд': number, 'Филиалов': actual_count, 'Проверка': 'OK'})
pd.DataFrame(test_summary)


## Необязательный просмотр через PowerPoint
Вся реализация локального рендера также находится в notebook. Она требует Windows и установленного PowerPoint, открывает PPTX только для чтения, проверяет рамки текста и высоту числовых подписей. Можно экспортировать все слайды или выбранные номера (`slides=[2]` для образца). Пути передаются через переменные окружения, не вставляются в код PowerShell.
По умолчанию `RENDER_PREVIEWS=False`. При включении PNG и JSON создаются во временной папке, картинки показываются в notebook, папка затем удаляется. Перед сохранением очищайте outputs. Постоянные PNG и подпапки output не создаются.


In [ ]:
def render_presentation(presentation, directory, *, slides=None):
    """Render locally via installed PowerPoint; no external project scripts."""
    source = Path(presentation).resolve(strict=True)
    target = Path(directory).resolve()
    if target.exists():
        raise FileExistsError('Выберите новый каталог превью.')
    if slides is not None and (not slides or any(not isinstance(n, int) or n < 1 for n in slides)):
        raise ValueError('Номера слайдов должны быть положительными целыми числами.')
    script = r"""
$ErrorActionPreference = 'Stop'
$source = (Resolve-Path -LiteralPath $env:PAPAGUN_RENDER_SOURCE).Path
$beforeHash = (Get-FileHash -LiteralPath $source -Algorithm SHA256).Hash
$target = [System.IO.Path]::GetFullPath($env:PAPAGUN_RENDER_DIR)
if (Test-Path -LiteralPath $target) { throw 'Preview directory already exists; select another Directory.' }
[System.IO.Directory]::CreateDirectory($target) | Out-Null
$app = $null
$deck = $null
$results = @()
try {
    $app = New-Object -ComObject PowerPoint.Application
    $deck = $app.Presentations.Open($source, -1, 0, 0)
    $indices = if ($env:PAPAGUN_RENDER_SLIDES) { @($env:PAPAGUN_RENDER_SLIDES.Split(',') | ForEach-Object { [int]$_ }) } else { @(1..$deck.Slides.Count) }
    foreach ($index in $indices) {
        if ($index -lt 1 -or $index -gt $deck.Slides.Count) { throw 'Invalid slide number.' }
        $slide = $deck.Slides.Item($index)
        $slide.Export((Join-Path $target "slide$index.png"), 'PNG', 1920, 1080)
        $overflow = @()
        $wrappedLabels = @()
        foreach ($shape in $slide.Shapes) {
            if ($shape.HasTextFrame -eq -1 -and $shape.TextFrame2.HasText -eq -1) {
                if ($shape.TextFrame2.TextRange.BoundWidth -gt ($shape.Width + 1) -or
                    $shape.TextFrame2.TextRange.BoundHeight -gt ($shape.Height + 1)) {
                    $overflow += [int]$shape.Id
                }
            }
            if ($shape.HasChart -eq -1) {
                $chart = $shape.Chart
                for ($seriesIndex = 1; $seriesIndex -le $chart.SeriesCollection().Count; $seriesIndex++) {
                    $series = $chart.SeriesCollection($seriesIndex)
                    for ($pointIndex = 1; $pointIndex -le $series.Points().Count; $pointIndex++) {
                        $point = $series.Points($pointIndex)
                        if ($point.HasDataLabel -and $point.DataLabel.Height -gt 24) {
                            $wrappedLabels += [pscustomobject]@{ series = $seriesIndex; point = $pointIndex }
                        }
                    }
                }
            }
        }
        $results += [pscustomobject]@{ slide = $index; text_overflow_shape_ids = @($overflow); wrapped_data_labels = @($wrappedLabels) }
        Write-Output "Rendered slide $index."
    }
    $results | ConvertTo-Json -Depth 5 | Set-Content -LiteralPath (Join-Path $target 'render_checks.json') -Encoding UTF8
    if (@($results | Where-Object { $_.text_overflow_shape_ids.Count -gt 0 -or $_.wrapped_data_labels.Count -gt 0 }).Count -gt 0) {
        throw 'Text overflow or wrapped numeric labels detected; inspect render_checks.json.'
    }
} finally {
    if ($null -ne $deck) { $deck.Close(); [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($deck) }
    if ($null -ne $app) { $app.Quit(); [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($app) }
}
if ((Get-FileHash -LiteralPath $source -Algorithm SHA256).Hash -ne $beforeHash) {
    throw 'Rendering changed the presentation.'
}
Write-Output 'PASS: PNG previews; presentation SHA-256 unchanged.'
"""
    environment = dict(os.environ, PAPAGUN_RENDER_SOURCE=str(source),
                       PAPAGUN_RENDER_DIR=str(target),
                       PAPAGUN_RENDER_SLIDES='' if slides is None else ','.join(map(str, slides)))
    completed = subprocess.run(
        ['powershell', '-NoProfile', '-NonInteractive', '-ExecutionPolicy', 'Bypass', '-Command', script],
        env=environment, capture_output=True, text=True, encoding='utf-8', errors='replace',
        creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0),
    )
    if completed.returncode:
        raise RuntimeError('PowerPoint не завершил проверку; убедитесь, что программа установлена и PPTX доступен.')
    report = json.loads((target / 'render_checks.json').read_text(encoding='utf-8-sig'))
    if isinstance(report, dict):
        report = [report]
    require(all(not r['text_overflow_shape_ids'] and not r['wrapped_data_labels'] for r in report),
            'PowerPoint detected text overflow or wrapped numbers.')
    return report


In [ ]:
render_summary = []
if RENDER_PREVIEWS:
    if test_pptx_path is None:
        raise ValueError('Сначала включите GENERATE_PRESENTATIONS и выполните экспорт.')
    from IPython.display import Image, display
    with TemporaryDirectory(prefix='papagun-preview-') as preview_directory:
        preview_path = Path(preview_directory) / 'slides'
        render_summary = render_presentation(test_pptx_path, preview_path)
        for result in render_summary:
            display(Image(filename=str(preview_path / f"slide{result['slide']}.png")))
